# Fase 2 · Almacenamiento y computación distribuida  (RA2 · UD2)

**Reto:** la dirección teme que la plataforma no aguante cuando se conecten todas las fuentes del puerto.
Tenéis que **demostrar con experimentos** que el clúster almacena y procesa grandes volúmenes, que tolera fallos
y que crece añadiendo nodos. Cada experimento debe quedar documentado: hipótesis → procedimiento (comandos) →
evidencias (capturas, tiempos) → conclusión.

| Parte | Experimento | CE |
|---|---|---|
| A | Cómo reparte HDFS los datos (bloques y réplicas) | 2.a |
| B | El modelo MapReduce y su equivalente en Spark | 2.b |
| C | Escalado: 1, 2 y 4 cores (y más tras añadir un worker) | 2.b, 2.e |
| D | Tolerancia a fallos en HDFS y en Spark | 2.c |
| E | «Guardar ahora, decidir después» (schema-on-read) | 2.d |
| F | Crecer: añadir un DataNode y reequilibrar | 2.e |

Algunos pasos se hacen en **PowerShell** (bloques ```powershell```). Copiad su salida en la memoria.

In [ ]:
from pyspark.sql import functions as F, Window
from puerto.sesion import crear_spark, reiniciar_spark, cronometro, executors_vivos, sello
PAREJA = "00"   # ✏️ vuestro número de pareja: su salida debe verse en las capturas de la memoria
sello(PAREJA)
from puerto import hdfs_web as h
import pandas as pd, matplotlib.pyplot as plt, os, time
os.makedirs("/workspace/entregas/F2", exist_ok=True)
spark = crear_spark("F2-distribuido")

## A · Bloques y réplicas
El bloque de HDFS en este clúster es de **32 MB** y cada bloque tiene **3 réplicas** (ver `docker/hadoop.env`).

In [ ]:
for d in h.datanodes():
    print(d)
ais_ficheros = h.listar("/puerto/raw/ais")
mayor = max(ais_ficheros, key=lambda f: f["bytes"])
ruta_mayor = "/puerto/raw/ais/" + mayor["nombre"]
print("Fichero AIS más grande:", ruta_mayor, round(mayor["bytes"] / 2**20, 1), "MB")
print(h.fsck(ruta_mayor)[:3000])

```powershell
.\bda.ps1 hdfs dfsadmin -report          # capacidad y estado de cada DataNode
.\bda.ps1 hdfs fsck /puerto -blocks | Select-Object -Last 25
```
**✏️ TAREA A1.** Con el `fsck` anterior: ¿cuántos bloques tiene el fichero?, ¿en qué DataNodes está cada réplica?
Dibujad (a mano o con draw.io) el reparto de bloques en los 3 DataNodes.

**✏️ TAREA A2.** Escribid el mismo fichero con **bloques de 8 MB** y **2 réplicas** y comparad `fsck` y `resumen`.

In [ ]:
conf = spark.sparkContext._jsc.hadoopConfiguration()
conf.set("dfs.blocksize", str(8 * 1024 * 1024))
conf.set("dfs.replication", "2")
(spark.read.text(ruta_mayor).coalesce(1)
     .write.mode("overwrite").text("/puerto/experimentos/ais_bloque8mb_rep2"))
conf.set("dfs.blocksize", str(32 * 1024 * 1024)); conf.set("dfs.replication", "3")   # volver a los valores normales
print(h.resumen("/puerto/experimentos/ais_bloque8mb_rep2"))
print(h.fsck("/puerto/experimentos/ais_bloque8mb_rep2", ubicaciones=False)[-1500:])

## B · El modelo MapReduce
MapReduce divide el trabajo en **map** (cada nodo procesa su trozo), **shuffle** (se agrupan las claves) y
**reduce** (se combinan). Ejecutad el clásico *wordcount* de Hadoop sobre las incidencias del puerto
(MapReduce en modo local dentro del NameNode):

```powershell
.\bda.ps1 hdfs dfs -rm -r -f /puerto/experimentos/wordcount
docker compose exec namenode bash -c "hadoop jar /opt/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar wordcount /puerto/raw/negocio/incidencias.jsonl /puerto/experimentos/wordcount"
.\bda.ps1 hdfs dfs -cat /puerto/experimentos/wordcount/part-r-00000 | Select-Object -First 30
```
Ahora lo mismo con Spark, primero al estilo MapReduce (RDD) y después con DataFrames:

In [ ]:
lineas = spark.sparkContext.textFile("/puerto/raw/negocio/incidencias.jsonl")
t = {}
with cronometro("wordcount RDD", t):
    conteo = (lineas.flatMap(lambda l: l.lower().split())       # MAP: cada línea -> palabras
                    .map(lambda p: (p, 1))                      # MAP: palabra -> (palabra, 1)
                    .reduceByKey(lambda a, b: a + b))           # SHUFFLE + REDUCE
    top = conteo.takeOrdered(15, key=lambda kv: -kv[1])
print(top)

In [ ]:
with cronometro("wordcount DataFrame", t):
    (spark.read.json("/puerto/raw/negocio/incidencias.jsonl")
          .select(F.explode(F.split(F.lower("texto"), r"\s+")).alias("palabra"))
          .groupBy("palabra").count().orderBy(F.desc("count")).show(15))

**✏️ TAREA B1.** Abrid http://localhost:4040 mientras se ejecuta y localizad el *job*, sus *stages* y sus *tasks*.
¿Dónde está el *shuffle*? ¿Cuántas tareas hubo en cada etapa y por qué? Explicad en la memoria las diferencias entre
MapReduce (disco entre fases) y Spark (memoria, DAG, evaluación perezosa).

## C · ¿Más recursos = más rápido?
Usamos una consulta pesada: distancia recorrida por cada buque y día a partir de las posiciones AIS
(fórmula de *haversine* con la posición anterior). Para que haya «volumen» multiplicamos los datos por `FACTOR`.
La ejecutamos limitando la aplicación a 1, 2 y 4 cores.

In [ ]:
FACTOR = 4          # subid a 8 si vuestro PC va sobrado; bajad a 2 si va justo

def consulta_pesada(spark, factor=FACTOR):
    ais = spark.read.parquet("/puerto/bronze/ais").filter("lat BETWEEN 36 AND 38 AND lon BETWEEN -8 AND -6")
    ais = ais.withColumn("copia", F.explode(F.sequence(F.lit(1), F.lit(factor))))
    w = Window.partitionBy("copia", "mmsi", "fecha").orderBy("ts")
    r = 6371.0
    d = (ais.withColumn("lat0", F.lag("lat").over(w)).withColumn("lon0", F.lag("lon").over(w))
            .withColumn("km", 2 * r * F.asin(F.sqrt(
                F.pow(F.sin(F.radians(F.col("lat") - F.col("lat0")) / 2), 2) +
                F.cos(F.radians("lat0")) * F.cos(F.radians("lat")) *
                F.pow(F.sin(F.radians(F.col("lon") - F.col("lon0")) / 2), 2)))))
    return d.groupBy("copia", "mmsi", "fecha").agg(F.sum("km").alias("km")).count()

tiempos = []
for cores in [1, 2, 4]:
    spark = reiniciar_spark(spark, nombre=f"F2-escalado-{cores}", cores_max=cores)
    consulta_pesada(spark, 1)                     # calentamiento (JVM, caché de metadatos)
    t0 = time.perf_counter(); consulta_pesada(spark); seg = time.perf_counter() - t0
    tiempos.append({"cores": cores, "segundos": round(seg, 1)})
    print(cores, "cores ->", round(seg, 1), "s")
escalado = pd.DataFrame(tiempos)
escalado["speedup"] = (escalado.segundos.iloc[0] / escalado.segundos).round(2)
escalado["eficiencia"] = (escalado.speedup / escalado.cores).round(2)
escalado.to_csv("/workspace/entregas/F2/escalado.csv", index=False)
escalado

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(escalado.cores, escalado.speedup, "o-", label="speedup medido")
ax.plot(escalado.cores, escalado.cores, "--", color="grey", label="ideal (lineal)")
ax.set_xlabel("cores"); ax.set_ylabel("speedup"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("/workspace/entregas/F2/speedup.png", dpi=120); plt.show()

**✏️ TAREA C1.** Interpretad la curva con la **ley de Amdahl**: ¿qué parte del trabajo no se paraleliza?
¿Por qué 4 cores no son 4 veces más rápidos en un solo PC? ¿Qué pasaría en un clúster real con varias máquinas?

**✏️ TAREA C2.** Mirad `spark.sql.shuffle.partitions` (8 por defecto en este clúster). Repetid la consulta con 4 cores y
valores 2, 8 y 64. Mostrad `explain()` y explicad el efecto de las particiones en el rendimiento.

In [ ]:
# ✏️ TAREA C2

## D · Tolerancia a fallos
### D1 · Se cae un DataNode
1. Lanzad una lectura larga en la celda siguiente.
2. **Mientras se ejecuta**, en PowerShell: `docker stop bda-puerto-datanode-2`
3. Observad que la lectura **termina bien** (el cliente usa otra réplica).
4. Esperad ~90 s y mirad http://localhost:9870 → *Datanodes*: el nodo pasa a *Dead*.
   `.\bda.ps1 hdfs fsck /puerto` mostrará bloques *under-replicated* y, al rato, HDFS crea réplicas nuevas.
5. `docker start bda-puerto-datanode-2` y volved a ejecutar `fsck`: ¿qué pasa con las réplicas sobrantes?

In [ ]:
with cronometro("lectura con fallo de DataNode"):
    for i in range(6):                                   # varias pasadas para que dé tiempo a parar el nodo
        n = spark.read.text("/puerto/raw/ais").count()
        print(f"pasada {i+1}: {n} líneas leídas")
print(h.fsck("/puerto/raw/ais", bloques=False, ubicaciones=False)[-900:])

### D2 · Se cae un worker de Spark
1. Lanzad la consulta pesada (siguiente celda) con 4 cores.
2. A los ~10 s, en PowerShell: `docker stop bda-puerto-spark-worker-2`
3. En http://localhost:4040 veréis tareas fallidas (*failed*) que se **reintentan** en el otro worker.
   El resultado es el mismo gracias al **linaje** (Spark sabe recalcular lo perdido).
4. `docker start bda-puerto-spark-worker-2`

In [ ]:
spark = reiniciar_spark(spark, nombre="F2-fallo-worker", cores_max=4)
with cronometro("consulta con caída de worker"):
    print("grupos:", consulta_pesada(spark))

**✏️ TAREA D1.** Documentad ambos experimentos con capturas (interfaz del NameNode, `fsck`, pestaña *Stages* de
Spark) y medid: tiempo hasta que el NameNode declara muerto el nodo y tiempo hasta recuperar la replicación completa.
¿Qué pasaría si se cayeran **3** DataNodes a la vez? ¿Y el NameNode? (investigad *NameNode HA* y *JournalNodes*).

## E · Guardar ahora, decidir después (*schema-on-read*)
Guardamos todas las posiciones AIS **en crudo** sin saber aún para qué servirían. Hoy aparece una pregunta nueva:
**¿cuánto tiempo esperan los buques fondeados en la bahía antes de atracar?** No hay que re-ingerir nada: leemos los
datos originales con el esquema que ahora necesitamos.

In [ ]:
from pyspark.sql import types as T
esquema_nuevo = "mmsi STRING, ts TIMESTAMP, lat DOUBLE, lon DOUBLE, estado_nav INT"
fondeo = (spark.read.schema(esquema_nuevo).json("/puerto/raw/ais")
          .filter((F.col("estado_nav") == 1) & F.col("lat").between(37.03, 37.09) & F.col("lon").between(-7.02, -6.94)))
horas_fondeo = (fondeo.groupBy("mmsi", F.to_date("ts").alias("dia"))
                .agg(((F.max("ts").cast("long") - F.min("ts").cast("long")) / 3600).alias("horas")))
horas_fondeo.agg(F.count("*").alias("episodios"), F.round(F.avg("horas"), 2).alias("media_h"),
                 F.round(F.expr("percentile_approx(horas, 0.9)"), 2).alias("p90_h")).show()

**✏️ TAREA E1.** Proponed **otra** pregunta nueva que se pueda contestar con los datos crudos ya guardados y resolvedla.
Comparad *schema-on-read* (lago de datos) con *schema-on-write* (base de datos relacional): ventajas, riesgos
(«pantano de datos»), coste de almacenar todo y papel de las cuotas de HDFS (`hdfs dfsadmin -setSpaceQuota`).

In [ ]:
# ✏️ TAREA E1

## F · El clúster crece
```powershell
.\bda.ps1 escalar datanode 4              # añade un cuarto DataNode
.\bda.ps1 hdfs dfsadmin -report          # ¿cuántos bloques tiene el nuevo?
docker compose exec namenode hdfs balancer -threshold 5
.\bda.ps1 hdfs dfsadmin -report          # ¿y ahora?
.\bda.ps1 escalar spark-worker 3          # añade un tercer worker (2 cores más)
```
**✏️ TAREA F1.** Documentad el antes/después del reparto de bloques y repetid el experimento C con **6 cores**.
Comparad este crecimiento «añadiendo módulos» con el de un servicio en la nube (Dataproc, EMR, Databricks):
¿qué cambia en coste y en tiempo de puesta en marcha?

In [ ]:
# ✏️ TAREA F1 (tras escalar): repetid la medición con cores_max=6

In [ ]:
spark.stop()